# CineInfini — Integration tests

End-to-end audit on a synthetic video. Validates that the full
pipeline (config → orchestrator → modules → aggregators → exporters)
composes correctly, not just individual units.

This notebook **does not require any external download** — it generates
its own test video on the fly.


## 1. Setup

In [1]:
import os, sys, tempfile
from pathlib import Path

repo = os.environ.get('CINEINFINI_REPO', '.')
sys.path.insert(0, f'{repo}/src')

import cineinfini
print(f"CineInfini {cineinfini.__version__}")

import cineinfini.modules  # registers all modules
from cineinfini.core.registry import all_modules
print(f"Registered modules: {len(all_modules())}")


CineInfini 0.4.8.4
Registered modules: 21


## 2. Generate a synthetic test video

In [2]:
import numpy as np
import cv2

work = Path(tempfile.mkdtemp(prefix='cineinfini_test_'))
video_path = work / 'synth.mp4'

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(str(video_path), fourcc, 24.0, (320, 180))
n_frames = 48  # 2 seconds @ 24fps
for i in range(n_frames):
    f = np.full((180, 320, 3), 30, dtype=np.uint8)
    # Moving rectangle (left to right)
    x = 20 + (i * 6) % 250
    f[60:120, x:x + 40] = (220, 220, 50)
    writer.write(f)
writer.release()

print(f"Wrote {video_path} ({n_frames} frames @ 24fps)")
print(f"File size: {video_path.stat().st_size} bytes")


Wrote /tmp/cineinfini_test_zz5m_5_q/synth.mp4 (48 frames @ 24fps)
File size: 10462 bytes


## 3. Audit it with the ultralight profile

In [3]:
from cineinfini.core.config import default_config, set_config

cfg = default_config()
# Mimic ultralight profile
cfg.processing['n_frames_per_shot'] = 4
cfg.processing['max_duration_s'] = 30
for mod_id in cfg.modules:
    cfg.modules[mod_id]['enabled'] = mod_id in (
        'motion_coherence', 'identity_consistency', 'semantic_consistency',
    )
cfg.paths['reports_dir'] = str(work / 'reports')
set_config(cfg)

from cineinfini.pipeline.orchestrator import run_audit
audit_data, output_dir = run_audit(video_path, output_dir=work / 'audit_out')
print(f"Output: {output_dir}")
print(f"Shots detected: {len(audit_data.get('gates') or {})}")
print(f"Modules run:   {list(audit_data.get('modules') or {})}")


  First pass: computing histogram differences (step=2)...
      First pass: 48/48 frames processed (elapsed 0.0s)
  [TIMING] First pass: 0.0s, 23 diffs
  Adaptive threshold: 0.080
  Second pass: detecting boundaries...


      Second pass: 48/48 frames processed (found 3 boundaries, elapsed 0.1s)
  [TIMING] Second pass: 0.1s, found 4 cut candidates
  [TIMING] Shot detection total: 0.1s, 1 shots generated
  Extracting 4 required frames...


      Extraction progress: 4/4 frames extracted
  [TIMING] extract_shot_frames_global: 0.064s, 4 unique frames


semantic_consistency: CLIP unavailable (No module named 'clip')


Output: /tmp/cineinfini_test_zz5m_5_q/audit_out
Shots detected: 1
Modules run:   ['motion_coherence', 'identity_consistency', 'semantic_consistency']


## 4. Compute VideoScore-style 5-axis fusion

In [4]:
from cineinfini.aggregators import attach_videoscore_to_audit

attach_videoscore_to_audit(audit_data)
print("=== VideoScore-style fusion ===")
for axis, value in audit_data['videoscore_axes'].items():
    s = f"{value:.3f}" if value is not None else "n/a"
    print(f"  {axis:30s} {s}")
print(f"  {'composite_score':30s} "
      f"{audit_data['composite_score']:.3f}" if audit_data['composite_score']
      else "composite: n/a")


=== VideoScore-style fusion ===
  visual_quality                 n/a
  temporal_consistency           0.926
  dynamic_degree                 1.000
  text_to_video_alignment        n/a
  factual_consistency            n/a
  composite_score                0.963


## 5. Export VBench-compatible JSON

In [5]:
from cineinfini.io.exporters import export_vbench_json

vbench_path = work / 'audit.vbench.json'
export_vbench_json(audit_data, vbench_path)

import json
payload = json.loads(vbench_path.read_text())
print(f"Model:               {payload['model']}")
print(f"Video:               {payload['video']}")
print(f"Measured (7):        {payload['measured_dimensions']}")
print(f"Unmeasured (9):      {payload['unmeasured_dimensions']}")
print()
print("Quality dimensions:")
for d in payload['measured_dimensions']:
    v = payload['scores'][d]
    print(f"  {d:30s} {v:.3f}")


Model:               CineInfini
Video:               unknown
Measured (7):        ['background_consistency', 'dynamic_degree']
Unmeasured (9):      ['subject_consistency', 'temporal_flickering', 'motion_smoothness', 'aesthetic_quality', 'imaging_quality', 'object_class', 'multiple_objects', 'human_action', 'color', 'spatial_relationship', 'scene', 'appearance_style', 'temporal_style', 'overall_consistency']

Quality dimensions:
  background_consistency         0.926
  dynamic_degree                 1.000


## 6. Cleanup

In [6]:
import shutil
shutil.rmtree(work, ignore_errors=True)
print(f"Removed {work}")


Removed /tmp/cineinfini_test_zz5m_5_q


## What this notebook validates

End-to-end:
1. ✅ Config loads + module registration works
2. ✅ Synthetic video is decoded and shot-detected
3. ✅ All 3 default-on modules produce per-shot scores
4. ✅ VideoScore aggregator composes 5 axes + composite from those scores
5. ✅ VBench exporter maps to 16 dimensions (7 measured, 9 null-by-design)
6. ✅ Output JSON is valid and machine-readable

This is the same pipeline that runs when you call `cineinfini audit`
from the CLI — minus shot-detection on real videos and minus the
heavier modules.
